In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/processed/engine.db")
train = pd.read_sql("SELECT * FROM features WHERE split = 'train'", con)

RUL_CAP = 125
train["rul_capped"] = train["rul"].clip(upper=RUL_CAP)

train[train["unit"] == 1][["cycle", "rul", "rul_capped"]].iloc[[0, 50, 66, 100, 191]]

,cycle,rul,rul_capped
0,1,191,125
50,51,141,125
66,67,125,125
100,101,91,91
191,192,0,0


In [2]:
feature_cols = [c for c in train.columns if c.startswith("s_")]

stats = train[feature_cols].agg(["min", "max"])
train_scaled = train.copy()
train_scaled[feature_cols] = (train[feature_cols] - stats.loc["min"]) / (
    stats.loc["max"] - stats.loc["min"]
)

print("Number of feature columns:", len(feature_cols))
train_scaled[feature_cols].describe().loc[["min", "max"]].round(2).T.head(6)

Number of feature columns: 28


,min,max
s_2,0.0,1.0
s_3,0.0,1.0
s_4,0.0,1.0
s_7,0.0,1.0
s_8,0.0,1.0
s_9,0.0,1.0


In [3]:
import json

stats.to_json("../data/processed/scaler_stats.json")

test = pd.read_sql("SELECT * FROM features WHERE split = 'test'", con)
test_scaled = test.copy()
test_scaled[feature_cols] = (test[feature_cols] - stats.loc["min"]) / (
    stats.loc["max"] - stats.loc["min"]
)

print("Feature columns:", len(feature_cols))
print("Test min:", test_scaled[feature_cols].min().min().round(3))
print("Test max:", test_scaled[feature_cols].max().max().round(3))

Feature columns: 28
Test min: -0.132
Test max: 1.081


In [4]:
train_scaled.to_csv("../data/processed/train_scaled.csv", index=False)
test_scaled.to_csv("../data/processed/test_scaled.csv", index=False)

print(train_scaled.shape, test_scaled.shape)
print(train_scaled[["unit", "cycle", "rul", "rul_capped"]].head(3))

(20631, 33) (13096, 32)
   unit  cycle  rul  rul_capped
0     1      1  191         125
1     1      2  190         125
2     1      3  189         125
